In [3]:
!pip install "tensorflow-text==2.19.*"

## **Importar dataset**

In [4]:
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
import re
import string

In [5]:
from google.colab import drive

drive.mount("/content/gdrive")

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [6]:
import pandas as pd

df_fake = pd.read_csv("/content/gdrive/MyDrive/ESCUELA/IRS/7MO/IA-2/Modulo-2.2/EVIDENCIA/xd/Fake.csv")
df_true = pd.read_csv("/content/gdrive/MyDrive/ESCUELA/IRS/7MO/IA-2/Modulo-2.2/EVIDENCIA/xd/True.csv")

In [7]:
df_fake.head(5)

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [8]:
df_true.head(5)

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [9]:
df_fake["class"] = 0
df_true["class"] = 1
df_fake.shape, df_true.shape

((23481, 5), (21417, 5))

In [10]:
df_fake = df_fake.sample(n=len(df_true), random_state=42)
df_fake.shape, df_true.shape

((21417, 5), (21417, 5))

In [11]:
df = pd.concat([df_fake, df_true], axis =0 )
df.head(10)

,title,text,subject,date,class
13474,ABOUT HILLARY’S COUGH: We Discovered The Secre...,,politics,"Jul 20, 2016",0
11994,BREAKING: OBAMACARE REPEAL Clears First Hurdle...,The Senate voted 51-48 this afternoon to proce...,politics,"Jan 4, 2017",0
19179,‘SLEEPY’ JUSTICE GINSBURG: Excites Crowd By Sa...,So much for the SCOTUS not being political Che...,left-news,"Feb 7, 2017",0
501,WATCH: Kellyanne Conway Very Upset Hillary Cl...,White House counselor Kellyanne Conway crawled...,News,"August 24, 2017",0
3492,"GOP Gives Trump The Middle Finger, Prepares T...",Donald Trump may have decided that Russia is g...,News,"December 9, 2016",0
1510,Trump Displays Incredible Ignorance Yet Again...,Have you ever wondered where a phrase started?...,News,"May 11, 2017",0
3296,Anthony Bourdain Reveals The ‘ONE Good Thing’...,While Donald Trump is currently freaking out b...,News,"December 22, 2016",0
17798,TRUMP HITS BACK After Cowgirl Congresswoman Tr...,The left is going ballistic over supposed word...,left-news,"Oct 18, 2017",0
9504,MEDIA DOWNPLAYS Attack By Unhinged Neighbor On...,"5 broken ribs with trouble breathing, lung con...",politics,"Nov 6, 2017",0
6087,Why This Attorney General Is Going After Trum...,New York Attorney General Eric Schneiderman is...,News,"May 31, 2016",0


##**Tranformaciones de los datos**

Vemos como esat estructurado el dataset

In [12]:
df.columns

Index(['title', 'text', 'subject', 'date', 'class'], dtype='object')

Vemos que tipo de datos tenemos y si hay valores nulos

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 42834 entries, 13474 to 21416
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    42834 non-null  object
 1   text     42834 non-null  object
 2   subject  42834 non-null  object
 3   date     42834 non-null  object
 4   class    42834 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 2.0+ MB


Contamos cuantos valores nulos tiene cada atributo

In [14]:
df.isnull().sum()

,0
title,0
text,0
subject,0
date,0
class,0


Quitamos valores nulos de los datos

In [15]:
df_clean = df.dropna()

Creamos un nuevo dataframe con los atributos que vamos a necesitar

In [16]:
df = df.drop(["text", "subject","date"], axis = 1)

Vemos como esta la estructura de este nuevo dataframe

In [17]:
df

,title,class
13474,ABOUT HILLARY’S COUGH: We Discovered The Secre...,0
11994,BREAKING: OBAMACARE REPEAL Clears First Hurdle...,0
19179,‘SLEEPY’ JUSTICE GINSBURG: Excites Crowd By Sa...,0
501,WATCH: Kellyanne Conway Very Upset Hillary Cl...,0
3492,"GOP Gives Trump The Middle Finger, Prepares T...",0
...,...,...
21412,'Fully committed' NATO backs new U.S. approach...,1
21413,LexisNexis withdrew two products from Chinese ...,1
21414,Minsk cultural hub becomes haven from authorities,1
21415,Vatican upbeat on possibility of Pope Francis ...,1


In [18]:
df.columns

Index(['title', 'class'], dtype='object')

##**Division del dataframe**

In [19]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df

,title,class
0,WATCH! Clueless Anti-Trump “Protesters” Asked ...,0
1,Investigators probe Trump knowledge of campaig...,1
2,Jeb Bush Kicks Off GOP Convention By Going Fu...,0
3,"TRUMP Puts Illegal Aliens, Un-vetted Muslim Im...",0
4,Trump Supporter Blows Up Fox Segment By Attac...,0
...,...,...
42829,WATCH: Dan Rather Explains How Close We Actua...,0
42830,MEGYN KELLY Reportedly Not Very Popular With F...,0
42831,"Iraq PM to visit Turkey on Wednesday, discuss ...",1
42832,"#BoycottPenzeys: HATEFUL, DIVISIVE Penzeys Spi...",0


In [20]:
def wordopt(text):
    text = text.lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub("\\W"," ",text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

<>:3: SyntaxWarning: invalid escape sequence '\['
<>:7: SyntaxWarning: invalid escape sequence '\w'
<>:3: SyntaxWarning: invalid escape sequence '\['
<>:7: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipython-input-593029456.py:3: SyntaxWarning: invalid escape sequence '\['
  text = re.sub('\[.*?\]', '', text)
/tmp/ipython-input-593029456.py:7: SyntaxWarning: invalid escape sequence '\w'
  text = re.sub('\w*\d\w*', '', text)


In [21]:
df["title"] = df["title"].apply(wordopt)

In [22]:
df

,title,class
0,watch clueless anti trump protesters asked ...,0
1,investigators probe trump knowledge of campaig...,1
2,jeb bush kicks off gop convention by going fu...,0
3,trump puts illegal aliens un vetted muslim im...,0
4,trump supporter blows up fox segment by attac...,0
...,...,...
42829,watch dan rather explains how close we actua...,0
42830,megyn kelly reportedly not very popular with f...,0
42831,iraq pm to visit turkey on wednesday discuss ...,1
42832,boycottpenzeys hateful divisive penzeys spi...,0


Definimos que la division sea train(70%), validation(15%), test(15%), nos aseguramos de darle un shuffle a las osbervaciones y que los 3 dataset esten balanceados.

In [23]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df["class"])

val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["class"])

print(f"Train: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}")

print("Distribución de clases:")
print(train_df["class"].value_counts(normalize=True).sort_index().round(3))
print(val_df["class"].value_counts(normalize=True).sort_index().round(3))
print(test_df["class"].value_counts(normalize=True).sort_index().round(3))


Train: 29983, Validation: 6425, Test: 6426
Distribución de clases:
class
0    0.5
1    0.5
Name: proportion, dtype: float64
class
0    0.5
1    0.5
Name: proportion, dtype: float64
class
0    0.5
1    0.5
Name: proportion, dtype: float64


Revisamos las dimensiones finales de cada uno

In [24]:
muestra = train_df.sample(10, random_state=42)

for _, row in muestra.iterrows():
    print("Question:", row["title"])
    print("Class:", row["class"])

Question: trump jr  tweet likening syrian refugees to poisoned skittles irks candy maker
Class: 1
Question: it begins  anthony scaramucci fires suspected leaker  anti trumper with ties to reince priebus
Class: 0
Question: trump attacks clinton on trade  says he should be handed victory
Class: 1
Question: update   states now giving obama middle finger on unlawful transgender bathroom decree
Class: 0
Question: lower taxes  big gains  the stocks poised to win from tax cuts
Class: 1
Question: elon musk s tesla and spacex oppose trump immigration order
Class: 1
Question: muslim miss universe contestant ignores competition rules other candidates must follow makes up her own rules
Class: 0
Question: iran bans u s  wrestlers in retaliation to trump s visa ban   tv
Class: 1
Question: senate to vote later on wednesday to work with house on tax bill  mcconnell
Class: 1
Question:  r i p  gop white house dreams  u s  jobs haven t grown this much since the  
Class: 0


In [25]:
class_names = ["fake_news","real_news"]

class_to_index = {name: i for i, name in enumerate(class_names)}
index_to_class = {i: name for i, name in enumerate(class_names)}

print(class_to_index)

{'fake_news': 0, 'real_news': 1}


##**Tokenización y vectorización de los datos**

En este caso la vectorización la vamos hacer con `'int'` para enumerar los tokens

In [26]:
from tensorflow.keras.layers import TextVectorization
MAX_SEQUENCE_LENGTH = 15
VOCAB_SIZE = 10000

int_vectorize_layer = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_SEQUENCE_LENGTH)

Separamos el texto y las etiquetas de cada dataset

In [27]:
texts = train_df["title"].astype(str).values
labels = train_df["class"].values

texts_test = test_df["title"].astype(str).values
labels_test = test_df["class"].values

texts_val = val_df["title"].astype(str).values
labels_val = val_df["class"].values

Ahora convertimos los datasets a tensorflow.data.Dataset y los agrupamos en batches

In [28]:
train_ds = tf.data.Dataset.from_tensor_slices((texts, labels)).batch(128)
test_ds = tf.data.Dataset.from_tensor_slices((texts_test, labels_test)).batch(128)
val_ds = tf.data.Dataset.from_tensor_slices((texts_val, labels_val)).batch(128)

Para sacar el vocab vamos a solo usar el dataset de train para que el modelo solo entrene con ese vocabulario

In [29]:
train_text = train_ds.map(lambda text, label: text)
int_vectorize_layer.adapt(train_text)

A partir de la tokenizacion de train, tokenizamos validation y test

In [30]:
def int_vectorize_text(text, label):
  text = tf.expand_dims(text, -1)
  return int_vectorize_layer(text), label

In [31]:
text_batch, label_batch = next(iter(train_ds))
first_question, first_label = text_batch[3], label_batch[3]
print("Title", first_question)
print("Label", first_label)

Title tf.Tensor(b' top scotus insider knows the  most likely  choice to replace scalia', shape=(), dtype=string)
Label tf.Tensor(0, shape=(), dtype=int64)


In [32]:
print("Original text:", first_question.numpy()[:200])
print("'int' vectorized question:",
      int_vectorize_text(first_question, first_label)[0])

Original text: b' top scotus insider knows the  most likely  choice to replace scalia'
'int' vectorized question: tf.Tensor(
[[ 107 1796 3388 1869    9  226  520  994    2 1400 1125    0    0    0
     0]], shape=(1, 15), dtype=int64)


In [33]:
print("1289 ---> ", int_vectorize_layer.get_vocabulary()[1289])
print("313 ---> ", int_vectorize_layer.get_vocabulary()[313])
print("Vocabulary size: {}".format(len(int_vectorize_layer.get_vocabulary())))

1289 --->  banned
313 --->  open
Vocabulary size: 10000


In [34]:
int_train_ds = train_ds.map(int_vectorize_text)
int_val_ds = val_ds.map(int_vectorize_text)
int_test_ds = test_ds.map(int_vectorize_text)

Con esto mantenemos los datos en memoria y prepara los datos del siguiente lote mientras entrena el actual

In [35]:
AUTOTUNE = tf.data.AUTOTUNE

def configure_dataset(dataset):
  return dataset.cache().prefetch(buffer_size=AUTOTUNE)

In [36]:
int_train_ds = configure_dataset(int_train_ds)
int_val_ds = configure_dataset(int_val_ds)
int_test_ds = configure_dataset(int_test_ds)

##**Importacion de datasets y vocabulario**

In [37]:
import os
import tensorflow as tf

base_dir = "/content/gdrive/MyDrive/ESCUELA/IRS/7MO/IA-2/Modulo-2.2/EVIDENCIA/xd"
os.makedirs(base_dir, exist_ok=True)

train_dir = os.path.join(base_dir, "train_dataset")
val_dir   = os.path.join(base_dir, "val_dataset")
test_dir  = os.path.join(base_dir, "test_dataset")

int_train_ds.save(train_dir)
int_val_ds.save(val_dir)
int_test_ds.save(test_dir)

print(f" Datasets guardados correctamente en:\n {base_dir}")

 Datasets guardados correctamente en:
 /content/gdrive/MyDrive/ESCUELA/IRS/7MO/IA-2/Modulo-2.2/EVIDENCIA/xd


Sacamos el vocabulario que obtuvimos del dataset de train

In [38]:
vocab = int_vectorize_layer.get_vocabulary()
print("Tamaño del vocabulario:", len(vocab))

Tamaño del vocabulario: 10000


In [39]:
with open(os.path.join(base_dir, "vocab.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(vocab))
print("Vocabulario guardado en vocab.txt")

Vocabulario guardado en vocab.txt
